# Lab Day 1 — Mạng nơ-ron và thí nghiệm huấn luyện (Forest CoverType)

**MSSV:** 2A202602715 · Chạy trên Google Colab (T4 GPU).

Cấu trúc: Part 0 (dữ liệu) → Part 1 (model + kiểm tra sức khoẻ) → Part 2 (pipeline, chọn lr, baseline 3 seed)
→ Part 3 (7 chủ đề thí nghiệm) → Part 4 (cấu hình cuối, eval, bảng, phân tích lỗi).

**Chống mất kết nối Colab:** ô đầu mount Google Drive; mọi `results/<exp_id>.json`, ảnh, bảng, dự đoán ghi thẳng vào
`MyDrive/lab_day1/submission_2A202602715/`. `run_cached` nạp lại lần chạy đã có trên Drive **nếu cfg trùng khớp hoàn toàn**
(đổi bất kỳ tham số nào sẽ tự chạy lại). Trọng số (`.pt`) chỉ lưu cho baseline và cấu hình cuối, ở `MyDrive/lab_day1/checkpoints/`
(ngoài thư mục nộp). Muốn huấn luyện lại tất cả: đặt `FORCE_RERUN = True`.

**Quy tắc công bằng:** cùng phép tách val (20%, phân tầng, seed 42), cùng 20 epoch, seed 1 cho mọi thí nghiệm; mỗi thí nghiệm
chỉ đổi một yếu tố so với baseline (ngoại lệ được ghi rõ). **Mọi lựa chọn chỉ dựa trên val.** Eval chỉ dùng ở Part 4.

In [ ]:
# ===== Đường dẫn, Drive và môi trường =====
import os, sys, json, math, copy, subprocess
IN_COLAB = "google.colab" in sys.modules
MSSV = "2A202602715"
REPO_URL = "https://github.com/kascenite/K4-Track4-Day1-Neuralnetwork.git"
QUICK = os.environ.get("LAB_QUICK") == "1"     # chỉ để tự kiểm tra code nhanh trên CPU; bản nộp chạy QUICK=False
FORCE_RERUN = False

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")                    # kết quả nằm trên Drive -> không mất khi Colab ngắt kết nối
    REPO_ROOT = "/content/K4-Track4-Day1-Neuralnetwork"
    if not os.path.exists(REPO_ROOT):
        subprocess.run(["git", "clone", "-q", REPO_URL, REPO_ROOT], check=True)
    else:
        subprocess.run(["git", "-C", REPO_ROOT, "pull", "-q"], check=True)
    os.chdir(f"{REPO_ROOT}/submission_{MSSV}/code")
    OUT_DIR = f"/content/drive/MyDrive/lab_day1/submission_{MSSV}"
    CKPT_DIR = "/content/drive/MyDrive/lab_day1/checkpoints"
else:
    REPO_ROOT = "../.."      # thư mục gốc repo (chứa data/ và scripts/), tính từ submission_<MSSV>/code/
    OUT_DIR = ".."           # = submission_<MSSV>/
    CKPT_DIR = f"{REPO_ROOT}/checkpoints"
OUT_DIR = os.environ.get("LAB_OUT", OUT_DIR)
CKPT_DIR = os.environ.get("LAB_CKPT", CKPT_DIR)
sys.path.insert(0, os.getcwd())
FIG, RES = f"{OUT_DIR}/figures", f"{OUT_DIR}/results"
for d in (FIG, RES, CKPT_DIR):
    os.makedirs(d, exist_ok=True)

import numpy as np, pandas as pd, torch
import matplotlib.pyplot as plt
from IPython.display import Image, display

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("torch", torch.__version__, "| device", device,
      "|", torch.cuda.get_device_name(0) if device.type == "cuda" else "CPU")
if IN_COLAB:
    assert device.type == "cuda", "Chưa bật GPU: Runtime -> Change runtime type -> T4 GPU"
print("REPO_ROOT =", os.path.abspath(REPO_ROOT), "\nOUT_DIR   =", os.path.abspath(OUT_DIR))

from data import prepare_data, N_NUMERIC
from model import MLP, EXPECTED_PARAMS, count_params, init_weights, activation_stats
from optimizer import build_optimizer, clip_gradients
from train import (DEFAULT_CFG, set_seed, evaluate, predict, run_experiment, run_cached, final_eval,
                   compute_loss, build_model)
from plots import plot_run, plot_compare
from results_table import save_result, load_results, to_row, write_xlsx

## Part 0 — Dữ liệu
`scripts/split_data.py` chia theo `split_metadata.csv` (không sửa). Validation = 20% của **train**, phân tầng, seed 42.
Chuẩn hoá 10 cột số bằng mean/std của **phần train còn lại**; 44 cột nhị phân giữ nguyên. Eval không tham gia bước nào.

In [ ]:
if not os.path.exists(f"{REPO_ROOT}/data/processed/eval.npz"):
    subprocess.run([sys.executable, "scripts/split_data.py"], cwd=REPO_ROOT, check=True)
data = prepare_data(device, val_fraction=0.2, seed=42, processed_dir=f"{REPO_ROOT}/data/processed")
if QUICK:   # tự kiểm tra nhanh: cắt nhỏ train/val (eval giữ nguyên để kiểm tra file dự đoán)
    data.update(X_tr=data["X_tr"][:20000], y_tr=data["y_tr"][:20000], X_val=data["X_val"][:5000], y_val=data["y_val"][:5000])

n_tr, n_val, n_ev = len(data["y_tr"]), len(data["y_val"]), len(data["y_eval"])
if not QUICK:
    assert (n_tr, n_val, n_ev) == (371_847, 92_962, 116_203)
props = pd.DataFrame({name: np.bincount(data[k].cpu().numpy(), minlength=7) / len(data[k])
                      for name, k in (("train", "y_tr"), ("val", "y_val"), ("eval", "y_eval"))})
print("Tỉ lệ lớp (%):"); display((100 * props).round(3))
num = data["X_tr"][:, :N_NUMERIC]
print("10 cột số trên X_tr: mean ∈ [%.2e, %.2e], std ∈ [%.4f, %.4f]" %
      (num.mean(0).min(), num.mean(0).max(), num.std(0).min(), num.std(0).max()))
y_val_np = data["y_val"].cpu().numpy()
majority = np.bincount(data["y_tr"].cpu().numpy()).argmax()
MAJ_ACC = float((y_val_np == majority).mean())
print(f"Mốc 'luôn đoán lớp đa số' trên val: acc = {MAJ_ACC:.4f}")

**Nhận xét Part 0:** kích thước đúng (371 847 / 92 962 / 116 203), tỉ lệ lớp ở train, val, eval gần như trùng nhau nhờ phân tầng,
10 cột số trên train có mean ≈ 0, std ≈ 1. Mốc thấp nhất phải vượt là accuracy "đoán lớp đa số" ≈ 0,4876 (macro-F1 của mốc này chỉ ≈ 0,094).

## Part 1 — Model và kiểm tra "sức khoẻ" ban đầu
`M-base` = `54 → 256 → 128 → 7`, ReLU, bias, **47 879 tham số**, logits `(B, 7)`, không softmax trong model, khởi tạo He (`kaiming_normal_`, bias = 0).

In [ ]:
set_seed(0)
model = MLP(hidden=(256, 128), dropout=0.0, init="he").to(device)
assert count_params(model) == EXPECTED_PARAMS[(256, 128)] == 47_879
print(model, "\nsố tham số:", count_params(model))

h = torch.randn(8, 54, device=device)
print("input", tuple(h.shape))
for layer in model.net:
    h = layer(h); print(f"  {layer.__class__.__name__:8s} -> {tuple(h.shape)}")
assert h.shape == (8, 7)

step0 = evaluate(model, data["X_val"], data["y_val"])
print(f"\nLoss bước 0 trên val = {step0['loss']:.4f}  (ln 7 = {math.log(7):.4f}, chênh {step0['loss'] - math.log(7):+.4f})")

In [ ]:
# Quá khớp 20 mẫu (dropout tắt, Adam lr 1e-2, 300 bước). Ảnh này chỉ hiện inline, không ghi vào figures/.
set_seed(0)
m20 = MLP(hidden=(256, 128), dropout=0.0, init="he").to(device)
opt20 = torch.optim.Adam(m20.parameters(), lr=1e-2)
X20, y20 = data["X_tr"][:20], data["y_tr"][:20]
curve = []
for step in range(300):
    m20.train()
    loss = compute_loss(m20(X20), y20, "ce")
    opt20.zero_grad(); loss.backward(); opt20.step()
    curve.append(loss.item())
acc20 = (predict(m20, X20) == y20).float().mean().item()
print(f"20 mẫu: loss đầu = {curve[0]:.4f}, loss cuối = {curve[-1]:.2e}, accuracy = {acc20:.2f}")
plt.figure(figsize=(5, 3)); plt.semilogy(curve); plt.xlabel("bước"); plt.ylabel("CE loss (log)")
plt.title("Quá khớp 20 mẫu"); plt.grid(alpha=.3); plt.show()
assert curve[-1] < 1e-2 and acc20 == 1.0

# Gradient chảy tới mọi tham số?
model.zero_grad()
compute_loss(model(data["X_tr"][:512]), data["y_tr"][:512], "ce").backward()
names = ["W1", "b1", "W2", "b2", "W3", "b3"]
for n, (pn, p) in zip(names, model.named_parameters()):
    print(f"  {n} ({pn:12s}) shape={tuple(p.shape)!s:12s} ‖grad‖ = {p.grad.norm().item():.4e}")
assert all(p.grad is not None and p.grad.norm() > 0 for p in model.parameters())

**Nhận xét Part 1:** <!--P1-->

## Part 2 — Pipeline và baseline
`run_experiment(cfg, data)` là hàm huấn luyện duy nhất; mọi thí nghiệm chỉ đổi dict `cfg`. Mỗi epoch ghi: train loss (chế độ eval,
tập con cố định 50 000 mẫu train), val loss/acc/macro-F1, `grad_norm` trước clip (TB và max), tỉ lệ bước bị clip, thời gian
(có `cuda.synchronize`), bộ nhớ cực đại, loss bước 0, best epoch theo val loss, cờ diverged.

**Chọn lr cho baseline (chỉ dùng val):** SGD+momentum 0,9 với lr ∈ {0,01; 0,03; 0,1; 0,3}, seed 1, 20 epoch. Chọn lr có
**val macro-F1 cao nhất** (ở best epoch). Các lần chạy này cũng là phần "SGD+momentum" của chủ đề bộ tối ưu.

*Dự đoán:* lr 0,01 hội tụ chậm (20 epoch chưa đủ); 0,3 có thể dao động mạnh vì momentum 0,9 khiến bước hiệu dụng ≈ lr/(1−μ) = 3;
lr tốt nhất nằm khoảng 0,03–0,1.

In [ ]:
EPOCHS = 2 if QUICK else 20
BASE = {**DEFAULT_CFG, "epochs": EPOCHS}
ALL = {}            # exp_id -> result (thứ tự = thứ tự dòng trong bảng)
NOTES = {}          # exp_id -> ghi chú (dự đoán / quan sát) cho cột notes

def run(exp_id, group, description, note="", ckpt=False, **changes):
    cfg = {**BASE, **changes, "exp_id": exp_id, "group": group, "description": description}
    r = run_cached(cfg, data, RES, CKPT_DIR if ckpt else None, force=FORCE_RERUN)
    plot_run(r, f"{FIG}/{exp_id}.png")
    ALL[exp_id] = r
    NOTES[exp_id] = note
    return r

def table(ids, ref=None):
    rows = []
    for i in ids:
        c, s = ALL[i]["cfg"], ALL[i]["summary"]
        rows.append(dict(exp_id=i, opt=c["optimizer"], lr=c["lr"], batch=c["batch"], hidden="-".join(map(str, c["hidden"])),
                         drop=c["dropout"], clip=c["clip_norm"], prec=c["precision"], init=c["init"], loss=c["loss"],
                         step0=s["step0_loss"], best_ep=s["best_epoch"], best_val_loss=s["best_val_loss"],
                         train_loss=s["final_train_loss"], val_loss=s["final_val_loss"], val_acc=s["val_acc"],
                         val_f1=s["val_macro_f1"], t_epoch=s["time_per_epoch_s"], mem_MB=s["peak_mem_MB"],
                         gn_p50=s["grad_norm_p50"], clip_frac=s["clip_frac"], diverged=s["diverged"]))
    df = pd.DataFrame(rows).set_index("exp_id")
    if ref is not None:
        df["Δf1_vs_base"] = df["val_f1"] - ref["f1_mean"]
        df["vượt 2σ?"] = df["Δf1_vs_base"].abs() > ref["f1_2sigma"]
    with pd.option_context("display.float_format", "{:.4f}".format, "display.width", 250, "display.max_columns", 30):
        display(df)
    return df

LR_GRID_SGDM = [0.01, 0.03, 0.1, 0.3]
for lr in LR_GRID_SGDM:
    run(f"opt-sgdm-lr{lr:g}", "optimizer", f"SGD+momentum 0.9, lr={lr:g} (dò lr cho baseline)", lr=lr,
        note="Dò lr baseline. Dự đoán: 0.01 chậm, 0.3 dao động, tốt nhất ~0.03-0.1.")
sweep = table([f"opt-sgdm-lr{lr:g}" for lr in LR_GRID_SGDM])
BASE_LR = float(sweep["val_f1"].idxmax().split("lr")[1])
BASE["lr"] = BASE_LR
print(f"==> lr baseline (chọn theo val macro-F1): {BASE_LR:g}")
plot_compare([ALL[f"opt-sgdm-lr{lr:g}"] for lr in LR_GRID_SGDM], ["val_loss", "val_macro_f1", "grad_norm"],
             f"{FIG}/compare_baseline_lr.png", "SGD+momentum: dò lr cho baseline (seed 1)")
display(Image(f"{FIG}/compare_baseline_lr.png"))

**Baseline 3 seed** (seed 1, 2, 3) ở lr đã chọn để đo độ nhiễu. Ngưỡng nhiễu = 2σ của val macro-F1 giữa các seed.
Trọng số tốt nhất (theo val loss) của baseline được lưu để dự đoán eval ở Part 4.

In [ ]:
SEEDS = [1, 2, 3]
for s in SEEDS:
    run(f"base-s{s}", "baseline", f"Baseline M-base, seed {s}", seed=s, ckpt=True,
        note="Baseline. Dự đoán: val acc >> 0.4876, đường cong còn giảm ở epoch 20 (chưa quá khớp).")
base_ids = [f"base-s{s}" for s in SEEDS]
bt = table(base_ids)
REF = dict(f1_mean=bt["val_f1"].mean(), f1_std=bt["val_f1"].std(ddof=1), acc_mean=bt["val_acc"].mean(),
           acc_std=bt["val_acc"].std(ddof=1), loss_mean=bt["best_val_loss"].mean(), loss_std=bt["best_val_loss"].std(ddof=1))
REF["f1_2sigma"] = 2 * REF["f1_std"]
print("Baseline val macro-F1 = %.4f ± %.4f | val acc = %.4f ± %.4f | best val loss = %.4f ± %.4f" %
      (REF["f1_mean"], REF["f1_std"], REF["acc_mean"], REF["acc_std"], REF["loss_mean"], REF["loss_std"]))
print("==> Ngưỡng nhiễu 2σ (val macro-F1) = %.4f" % REF["f1_2sigma"])
assert all(ALL[i]["summary"]["val_acc"] > MAJ_ACC for i in base_ids)
plot_compare([ALL[i] for i in base_ids], ["train_loss", "val_loss", "val_macro_f1"], f"{FIG}/compare_baseline.png",
             "Baseline: 3 seed (độ nhiễu)")
display(Image(f"{FIG}/compare_baseline.png")); display(Image(f"{FIG}/base-s1.png"))

**Nhận xét baseline:** <!--P2-->

## Part 3 — Thí nghiệm
Mỗi thí nghiệm: dự đoán trước → đổi một yếu tố so với `base-s1` (seed 1, cùng lr, 20 epoch) → chạy → đối chiếu, so với ngưỡng 2σ.

### 3.1 Hàm mất mát — CE vs MSE (`loss`)
MSE = `F.mse_loss(logits, one_hot(y))`, trung bình trên mọi phần tử B×7, không có hệ số 1/2.

**Dự đoán:** với CE, ∂L/∂z = softmax(z) − y: khi dự đoán sai nặng gradient vẫn lớn (≈ 1). MSE trên logit có ∂L/∂z = 2(z − y)/7 và
bị chia thêm cho 7 vì lấy trung bình trên 7 cột ⇒ gradient nhỏ hơn nhiều ở cùng lr ⇒ học chậm hơn, macro-F1 thấp hơn rõ, nhất là ở lớp hiếm.
Thử thêm MSE với lr ×3 để xem chênh lệch có phải chỉ do thang gradient. Không so giá trị loss (khác thang đo), chỉ so acc/macro-F1.

In [ ]:
run("loss-mse", "loss", "MSE trên one-hot thay cho CE (cùng lr)", loss="mse",
    note="Dự đoán: gradient nhỏ hơn CE -> học chậm, F1 thấp hơn.")
run("loss-mse-lr3x", "loss", "MSE, lr x3 so với baseline (đổi 2 yếu tố: loss + lr)", loss="mse", lr=3 * BASE_LR,
    note="Kiểm tra: tăng lr có bù được gradient nhỏ của MSE không. Đổi 2 yếu tố.")
table(["base-s1", "loss-mse", "loss-mse-lr3x"], REF)
plot_compare([ALL[i] for i in ["base-s1", "loss-mse", "loss-mse-lr3x"]], ["val_acc", "val_macro_f1", "grad_norm"],
             f"{FIG}/compare_loss.png", "CE vs MSE (so bằng metric, không so loss)")
display(Image(f"{FIG}/compare_loss.png"))

**Đối chiếu 3.1:** <!--P31-->

### 3.2 Bộ tối ưu hoá (`optimizer`)
SGD (không momentum) lr ∈ {0,03; 0,1; 0,3}; Adam lr ∈ {3e-4; 1e-3; 3e-3} (betas 0,9/0,999, eps 1e-8); AdamW cùng lưới lr, weight_decay 0,01.
SGD+momentum đã có ở Part 2. So mỗi bộ ở **lr tốt nhất của nó** (theo val macro-F1).

**Dự đoán:** SGD thuần cần lr lớn hơn SGD+momentum ~10 lần (momentum 0,9 khuếch đại bước ≈ 10×) và vẫn chậm nhất;
Adam/AdamW nhanh ở vài epoch đầu nhờ bước chuẩn hoá theo từng tham số, ở lr tốt nhất có thể nhỉnh hơn SGD+momentum.
AdamW và Adam gần như bằng nhau (wd 0,01 nhỏ, mô hình chưa quá khớp) — khác biệt có lẽ nằm trong nhiễu.

In [ ]:
OPT_GRID = {"sgd": [0.03, 0.1, 0.3], "adam": [3e-4, 1e-3, 3e-3], "adamw": [3e-4, 1e-3, 3e-3]}
TAG = {"sgd": "sgd", "sgd_momentum": "sgdm", "adam": "adam", "adamw": "adamw"}
for opt, lrs in OPT_GRID.items():
    for lr in lrs:
        wd = 0.01 if opt == "adamw" else 0.0
        run(f"opt-{TAG[opt]}-lr{lr:g}", "optimizer", f"{opt}, lr={lr:g}, weight_decay={wd:g}", optimizer=opt, lr=lr,
            weight_decay=wd, note=f"Dự đoán: {'SGD thuần chậm hơn SGD+momentum' if opt == 'sgd' else 'Adam-family nhanh ở epoch đầu'}.")
opt_ids = {"sgd_momentum": [f"opt-sgdm-lr{lr:g}" for lr in LR_GRID_SGDM],
           **{o: [f"opt-{TAG[o]}-lr{lr:g}" for lr in lrs] for o, lrs in OPT_GRID.items()}}
ot = table([i for ids in opt_ids.values() for i in ids], REF)
BEST_OPT = {o: max(ids, key=lambda i: ALL[i]["summary"]["val_macro_f1"]) for o, ids in opt_ids.items()}
print("Lr tốt nhất của từng bộ tối ưu (theo val macro-F1):", BEST_OPT)
table(list(BEST_OPT.values()), REF)
plot_compare([ALL[i] for i in BEST_OPT.values()], ["train_loss", "val_loss", "val_macro_f1"],
             f"{FIG}/compare_optimizer.png", "Mỗi bộ tối ưu ở lr tốt nhất của nó (seed 1)")
fig, ax = plt.subplots(figsize=(6, 4))
for o, ids in opt_ids.items():
    ax.semilogx([ALL[i]["cfg"]["lr"] for i in ids], [ALL[i]["summary"]["val_macro_f1"] for i in ids], "o-", label=o)
ax.axhspan(REF["f1_mean"] - REF["f1_2sigma"], REF["f1_mean"] + REF["f1_2sigma"], color="gray", alpha=.2, label="baseline ± 2σ")
ax.set(xlabel="lr (log)", ylabel="val macro-F1 (best epoch)", title="Độ nhạy theo lr của từng bộ tối ưu"); ax.grid(alpha=.3); ax.legend()
fig.savefig(f"{FIG}/compare_optimizer_lr.png", dpi=110, bbox_inches="tight"); plt.close(fig)
display(Image(f"{FIG}/compare_optimizer.png")); display(Image(f"{FIG}/compare_optimizer_lr.png"))

**Đối chiếu 3.2:** <!--P32-->

### 3.3 Hyper-parameter (`hparam`)
Batch 128 và 2048 (cùng lr ⇒ số bước/epoch lần lượt 2 906 và 182 so với 727); batch 2048 với lr ×4 (quy tắc tăng lr theo lô, không warmup — đổi 2 yếu tố);
độ rộng `M-wide` (512-256, 161 287 tham số); độ sâu `M-deep` (256-128-64, 55 687 tham số).

**Dự đoán:** cùng 20 epoch, batch 128 có gấp 4 số bước ⇒ val F1 cao hơn nhưng mỗi epoch chậm hơn nhiều (nhiều lần gọi kernel);
batch 2048 cùng lr có 1/4 số bước ⇒ kém hơn; tăng lr ×4 bù lại phần lớn. `M-wide` tăng năng lực ⇒ F1 cao hơn (mô hình đang chưa khớp);
`M-deep` thay đổi ít.

In [ ]:
run("hp-batch128", "hparam", "batch 128 (cùng lr)", batch=128, note="Dự đoán: nhiều bước hơn -> F1 cao hơn, chậm hơn/epoch.")
run("hp-batch2048", "hparam", "batch 2048 (cùng lr)", batch=2048, note="Dự đoán: ít bước -> F1 thấp hơn.")
run("hp-batch2048-lr4x", "hparam", "batch 2048, lr x4 (linear scaling; đổi 2 yếu tố)", batch=2048, lr=4 * BASE_LR,
    note="Dự đoán: lr x4 bù phần lớn số bước bị mất.")
run("hp-wide", "hparam", "M-wide 512-256", hidden=(512, 256), note="Dự đoán: tăng năng lực -> F1 cao hơn.")
run("hp-deep", "hparam", "M-deep 256-128-64", hidden=(256, 128, 64), note="Dự đoán: thay đổi nhỏ.")
hp_ids = ["base-s1", "hp-batch128", "hp-batch2048", "hp-batch2048-lr4x", "hp-wide", "hp-deep"]
ht = table(hp_ids, REF)
print("Số bước cập nhật mỗi epoch:", {i: ALL[i]["summary"]["steps_per_epoch"] for i in hp_ids})
plot_compare([ALL[i] for i in hp_ids], ["train_loss", "val_loss", "val_macro_f1"], f"{FIG}/compare_hparam.png",
             "Hyper-parameter: batch, độ rộng, độ sâu")
display(Image(f"{FIG}/compare_hparam.png"))

**Đối chiếu 3.3:** <!--P33-->

### 3.4 Dropout (`dropout`)
q ∈ {0,1; 0,3; 0,5}, đặt sau ReLU của 2 lớp ẩn. Train loss đo ở chế độ eval nên so sánh được với val.

**Dự đoán:** baseline gần như chưa quá khớp (khoảng cách val − train loss nhỏ, cả hai còn giảm ở epoch 20) nên dropout không giúp,
q càng lớn càng làm chậm học ⇒ val F1 giảm dần theo q; khoảng cách train–val co lại nhưng vì train loss tăng lên chứ không phải val giảm.

In [ ]:
for q in [0.1, 0.3, 0.5]:
    run(f"drop-{q:g}", "dropout", f"dropout q={q:g} sau mỗi ReLU", dropout=q,
        note="Dự đoán: mô hình chưa quá khớp -> dropout không giúp, F1 giảm theo q.")
drop_ids = ["base-s1", "drop-0.1", "drop-0.3", "drop-0.5"]
dt = table(drop_ids, REF)
print("Khoảng cách val − train loss (epoch cuối):", {i: round(ALL[i]["summary"]["final_val_loss"] - ALL[i]["summary"]["final_train_loss"], 4) for i in drop_ids})
plot_compare([ALL[i] for i in drop_ids], ["train_loss", "val_loss", "val_macro_f1"], f"{FIG}/compare_dropout.png",
             "Dropout: train loss (eval mode) vs val loss")
display(Image(f"{FIG}/compare_dropout.png"))

**Đối chiếu 3.4:** <!--P34-->

### 3.5 Gradient clipping (`clipping`)
Chọn `c` từ phân bố `grad_norm` theo bước của `base-s1`: `c` = trung vị (p50) ⇒ clipping kích hoạt ở ~một nửa số bước.
Thí nghiệm phản chứng: lr cao = 10 × lr baseline, **không clip** vs **có clip** (cùng `c`).

**Dự đoán:** ở lr bình thường, clip ở p50 chỉ làm bước nhỏ đi một chút ⇒ khác biệt nằm trong nhiễu (có thể hơi chậm hơn).
Ở lr ×10 không clip, SGD+momentum dao động mạnh/NaN (bước hiệu dụng rất lớn); có clip, độ dài bước bị chặn bởi lr·c nên huấn luyện
ổn định hơn, dù vẫn kém baseline.

In [ ]:
s1 = ALL["base-s1"]["summary"]
print("grad_norm theo bước của base-s1: p50=%.3f p90=%.3f p99=%.3f max=%.3f" %
      (s1["grad_norm_p50"], s1["grad_norm_p90"], s1["grad_norm_p99"], s1["grad_norm_max"]))
CLIP_C = round(s1["grad_norm_p50"], 3)
HIGH_LR = 10 * BASE_LR
run(f"clip-{CLIP_C:g}", "clipping", f"clip c={CLIP_C:g} (p50 grad_norm base-s1), lr baseline", clip_norm=CLIP_C,
    note="Dự đoán: ở lr thường khác biệt nằm trong nhiễu.")
run("clip-none-highlr", "clipping", f"lr x10 = {HIGH_LR:g}, không clip (đổi lr)", lr=HIGH_LR,
    note="Dự đoán: dao động mạnh hoặc NaN.")
run(f"clip-{CLIP_C:g}-highlr", "clipping", f"lr x10 = {HIGH_LR:g}, clip c={CLIP_C:g} (đổi lr + clip)", lr=HIGH_LR, clip_norm=CLIP_C,
    note="Dự đoán: clipping giữ huấn luyện ổn định ở lr cao.")
clip_ids = ["base-s1", f"clip-{CLIP_C:g}", "clip-none-highlr", f"clip-{CLIP_C:g}-highlr"]
table(clip_ids, REF)
plot_compare([ALL[i] for i in clip_ids], ["val_loss", "val_macro_f1", "grad_norm_max"], f"{FIG}/compare_clipping.png",
             f"Gradient clipping (c = {CLIP_C:g}); grad_norm đo trước khi clip")
display(Image(f"{FIG}/compare_clipping.png"))

**Đối chiếu 3.5:** <!--P35-->

### 3.6 Mixed precision (`amp`)
FP16: `autocast(float16)` + `GradScaler` (luôn `unscale_` trước khi đo/clip gradient). BF16: `autocast(bfloat16)`, không GradScaler.
Tham số vẫn FP32. Thêm `M-wide` + FP16 để so với `hp-wide` (FP32) xem mạng lớn hơn có hưởng lợi không.

**Dự đoán:** T4 có Tensor Core FP16 nhưng mạng 48k tham số và batch 512 quá nhỏ: thời gian bị chi phối bởi chi phí gọi kernel và
các phép cast thêm ⇒ FP16 **không nhanh hơn**, có thể chậm hơn; bộ nhớ đỉnh gần như không đổi (dữ liệu FP32 trên GPU chiếm phần lớn).
T4 (Turing) không có phần cứng BF16 ⇒ BF16 chậm hơn hoặc báo không hỗ trợ. Độ chính xác giữ nguyên trong nhiễu.

In [ ]:
bf16_ok = device.type == "cuda" and torch.cuda.is_bf16_supported()
print("torch.cuda.is_bf16_supported():", bf16_ok, "| capability:", torch.cuda.get_device_capability(0) if device.type == "cuda" else None)
run("amp-fp16", "amp", "FP16 autocast + GradScaler", precision="fp16", note="Dự đoán: không nhanh hơn với mạng nhỏ.")
amp_ids = ["base-s1", "amp-fp16"]
try:
    run("amp-bf16", "amp", "BF16 autocast (không GradScaler)", precision="bf16",
        note=f"T4 không có BF16 phần cứng; is_bf16_supported={bf16_ok}.")
    amp_ids.append("amp-bf16")
except Exception as e:      # ghi lý do vào notes nếu GPU không chạy được BF16
    print("BF16 lỗi:", repr(e))
run("amp-fp16-wide", "amp", "M-wide + FP16 (so với hp-wide FP32; đổi 2 yếu tố)", precision="fp16", hidden=(512, 256),
    note="So với hp-wide: mạng rộng hơn có hưởng lợi từ FP16 không.")
amp_ids += ["hp-wide", "amp-fp16-wide"]
table(amp_ids, REF)
plot_compare([ALL[i] for i in amp_ids], ["val_macro_f1", "epoch_time_s", "grad_norm"], f"{FIG}/compare_amp.png",
             "Mixed precision: FP32 vs FP16 vs BF16")
display(Image(f"{FIG}/compare_amp.png"))

**Đối chiếu 3.6:** <!--P36-->

### 3.7 Khởi tạo tham số (`init`)
`zeros` (W = 0), `normal` (N(0, 0,01²)), `xavier` (`xavier_normal_`, Var = 2/(n_in+n_out)), `default` (mặc định `nn.Linear`:
U(±1/√n_in), Var = 1/(3·n_in)) so với `he` (baseline, Var = 2/n_in). Bias = 0 (trừ `default`).

**Dự đoán:** `zeros`: mọi nơ-ron ẩn ra ReLU(0) = 0 ⇒ gradient tới W1, W2 bằng 0 và các nơ-ron đối xứng ⇒ chỉ bias lớp cuối học được ⇒
mô hình chỉ học được tần suất lớp (acc ≈ 0,4876, F1 ≈ 0,094). `normal` 0,01: kích hoạt co lại ~×0,1 mỗi lớp ⇒ logit ≈ 0, loss bước 0 = ln 7 đúng,
nhưng gradient lớp đầu nhỏ ⇒ khởi động chậm, sau 20 epoch có thể bắt kịp một phần. `xavier`/`default` cho std kích hoạt giảm dần qua lớp nhưng
mạng chỉ 3 lớp nên khác biệt với He nhỏ, có thể nằm trong nhiễu.

In [ ]:
xb = data["X_val"][:4096]
rows = {}
for ini in ["zeros", "normal", "xavier", "default", "he"]:
    set_seed(1)
    m = MLP(hidden=(256, 128), init=ini).to(device)
    rows[ini] = activation_stats(m, xb) + [evaluate(m, data["X_val"], data["y_val"])["loss"]]
print("Std kích hoạt ở bước 0 (một lô val 4096 mẫu), M-base:")
display(pd.DataFrame(rows, index=["std ReLU1", "std ReLU2", "std logits", "step0 val loss"]).T.round(4))

# Minh hoạ "30 lớp ReLU" như slide: chỉ khởi tạo + forward, KHÔNG huấn luyện (không phải kiến trúc nộp).
deep = {}
for ini in ["normal", "xavier", "default", "he"]:
    set_seed(1)
    m = MLP(hidden=(256,) * 30, init=ini).to(device)
    deep[ini] = activation_stats(m, xb)[:-1]
fig, ax = plt.subplots(figsize=(6, 3.5))
for ini, v in deep.items():
    ax.semilogy(range(1, 31), v, "o-", ms=3, label=ini)
ax.set(xlabel="lớp ReLU", ylabel="std kích hoạt (log)", title="30 lớp ReLU 256 nơ-ron: std kích hoạt ở bước 0"); ax.grid(alpha=.3); ax.legend()
plt.show()

for ini in ["zeros", "normal", "xavier", "default"]:
    run(f"init-{ini}", "init", f"khởi tạo {ini} thay cho He", init=ini, note=f"Dự đoán: xem markdown 3.7 ({ini}).")
init_ids = ["base-s1", "init-zeros", "init-normal", "init-xavier", "init-default"]
table(init_ids, REF)
plot_compare([ALL[i] for i in init_ids], ["train_loss", "val_macro_f1", "grad_norm"], f"{FIG}/compare_init.png",
             "Khởi tạo tham số")
display(Image(f"{FIG}/compare_init.png"))

**Đối chiếu 3.7:** <!--P37-->

## Part 4 — Cấu hình cuối, đánh giá trên eval, bảng
### 4.1 Chọn cấu hình cuối (chỉ bằng val; quy tắc viết TRƯỚC khi xem kết quả)
1. Bộ tối ưu + lr: lần chạy có val macro-F1 cao nhất trong nhóm `optimizer`.
2. Kiến trúc: `M-wide` hoặc `M-deep` nếu val macro-F1 của nó > trung bình baseline + 2σ (chọn cái cao hơn), ngược lại `M-base`.
3. Dropout: q tốt nhất nếu vượt baseline + 2σ, ngược lại 0. Batch, clipping, precision, init giữ như baseline
   (chỉ đổi nếu vượt 2σ: batch 128 nếu `hp-batch128` vượt).
4. Huấn luyện **40 epoch** (baseline chưa hội tụ ở epoch 20); best epoch chọn theo val loss (tương đương dừng sớm).
5. Chạy 3 seed; nộp dự đoán của seed có **val macro-F1 cao nhất**.

Eval chỉ chạy đúng 2 lần: `base-s1` và cấu hình cuối. Không chỉnh gì sau khi xem điểm eval.

In [ ]:
def beats(i):
    return ALL[i]["summary"]["val_macro_f1"] > REF["f1_mean"] + REF["f1_2sigma"]

best_opt_id = max(opt_ids_all := [i for ids in opt_ids.values() for i in ids], key=lambda i: ALL[i]["summary"]["val_macro_f1"])
bo = ALL[best_opt_id]["cfg"]
FINAL = dict(optimizer=bo["optimizer"], lr=bo["lr"], weight_decay=bo["weight_decay"])
arch = [i for i in ("hp-wide", "hp-deep") if beats(i)]
if arch:
    FINAL["hidden"] = tuple(ALL[max(arch, key=lambda i: ALL[i]["summary"]["val_macro_f1"])]["cfg"]["hidden"])
dropc = [i for i in ("drop-0.1", "drop-0.3", "drop-0.5") if beats(i)]
if dropc:
    FINAL["dropout"] = ALL[max(dropc, key=lambda i: ALL[i]["summary"]["val_macro_f1"])]["cfg"]["dropout"]
if beats("hp-batch128"):
    FINAL["batch"] = 128
FINAL["epochs"] = 2 if QUICK else 40
print("Tối ưu tốt nhất:", best_opt_id, "| kiến trúc vượt 2σ:", arch, "| dropout vượt 2σ:", dropc,
      "| batch128 vượt 2σ:", beats("hp-batch128"))
print("==> Cấu hình cuối (thay đổi so với baseline):", FINAL)

desc = "Cấu hình cuối: " + ", ".join(f"{k}={v}" for k, v in FINAL.items())
for s in SEEDS:
    run(f"final-s{s}", "final", desc + f", seed {s}", seed=s, ckpt=True, **FINAL,
        note="Cấu hình cuối chọn theo val (quy tắc 4.1).")
final_ids = [f"final-s{s}" for s in SEEDS]
ft = table(final_ids, REF)
FINAL_F1 = (ft["val_f1"].mean(), ft["val_f1"].std(ddof=1))
print("Cấu hình cuối val macro-F1 = %.4f ± %.4f (baseline %.4f ± %.4f)" % (*FINAL_F1, REF["f1_mean"], REF["f1_std"]))
SUBMIT_ID = ft["val_f1"].idxmax()
print("==> Nộp dự đoán của", SUBMIT_ID, "(val macro-F1 cao nhất trong 3 seed)")
plot_compare([ALL[i] for i in final_ids] + [ALL["base-s1"]], ["train_loss", "val_loss", "val_macro_f1"],
             f"{FIG}/compare_final.png", "Cấu hình cuối (3 seed) vs baseline")
display(Image(f"{FIG}/compare_final.png"))

### 4.2 Đánh giá trên eval (`scripts/evaluate.py`)
Baseline (`base-s1`) ghi ra `predictions_eval_baseline.csv` / `eval_result_baseline.json` (chỉ để so sánh);
cấu hình cuối ghi ra `predictions_eval.csv` / `eval_result.json` (file nộp).

In [ ]:
EVAL_BASE = final_eval(ALL["base-s1"]["cfg"], ALL["base-s1"], data, f"{OUT_DIR}/predictions_eval_baseline.csv",
                       repo_root=REPO_ROOT, out_json=f"{OUT_DIR}/eval_result_baseline.json")
EVAL_FINAL = final_eval(ALL[SUBMIT_ID]["cfg"], ALL[SUBMIT_ID], data, f"{OUT_DIR}/predictions_eval.csv",
                        repo_root=REPO_ROOT, out_json=f"{OUT_DIR}/eval_result.json")
pred = pd.read_csv(f"{OUT_DIR}/predictions_eval.csv")
assert list(pred.columns) == ["row_id", "pred"] and len(pred) == 116_203 and pred["row_id"].is_unique
print(pd.DataFrame({"base-s1": [ALL["base-s1"]["summary"]["val_macro_f1"], EVAL_BASE["macro_f1"], EVAL_BASE["accuracy"]],
                    SUBMIT_ID: [ALL[SUBMIT_ID]["summary"]["val_macro_f1"], EVAL_FINAL["macro_f1"], EVAL_FINAL["accuracy"]]},
                   index=["val macro-F1", "eval macro-F1", "eval accuracy"]).round(4))

In [ ]:
# Phân tích lỗi theo lớp (eval, cấu hình cuối) + số mẫu train từng lớp để lý giải
pc = pd.DataFrame(EVAL_FINAL["per_class"]).set_index("cls")
pc["train_count"] = np.bincount(data["y_tr"].cpu().numpy(), minlength=7)
display(pc.round(4))
cm = np.array(EVAL_FINAL["confusion_matrix"])
cmn = cm / cm.sum(1, keepdims=True)
fig, ax = plt.subplots(figsize=(5.5, 4.5))
im = ax.imshow(cmn, cmap="Blues")
for i in range(7):
    for j in range(7):
        ax.text(j, i, f"{cmn[i, j]:.2f}", ha="center", va="center", fontsize=7, color="white" if cmn[i, j] > .5 else "black")
ax.set(xlabel="dự đoán", ylabel="nhãn thật", title=f"Ma trận nhầm lẫn chuẩn hoá theo hàng (eval, {SUBMIT_ID})")
plt.colorbar(im); plt.show()
hard = int(pc["f1"].idxmin())
off = cm[hard].copy(); off[hard] = 0
conf_with = int(off.argmax())
print(f"Lớp khó nhất: {hard} (F1 = {pc.loc[hard, 'f1']:.4f}); bị nhầm nhiều nhất sang lớp {conf_with} "
      f"({off[conf_with]}/{cm[hard].sum()} = {off[conf_with] / cm[hard].sum():.1%})")
# So sánh đặc trưng của 2 lớp hay nhầm (trên TRAIN, giá trị đã chuẩn hoá): 10 cột số
Xtr, ytr = data["X_tr"].cpu().numpy(), data["y_tr"].cpu().numpy()
names = ["Elevation", "Aspect", "Slope", "H_Dist_Hydro", "V_Dist_Hydro", "H_Dist_Road", "Hillshade_9am",
         "Hillshade_Noon", "Hillshade_3pm", "H_Dist_Fire"]
display(pd.DataFrame({f"lớp {c}": Xtr[ytr == c, :10].mean(0) for c in (hard, conf_with)}, index=names).round(3))

**Phân tích lỗi:** <!--P4-->

### 4.3 Bảng `experiments.xlsx`
Mỗi lần chạy một dòng (cùng thứ tự chạy), sheet `Seeds` = `base-s1..3`, `eval_acc`/`eval_macro_f1` chỉ cho `base-s1` và cấu hình nộp.

In [ ]:
SUMMARY_NOTES = {}   # nhận xét ngắn theo nhóm (điền sau khi có kết quả)  <!--SUMMARY-->
rows = []
for i, r in ALL.items():
    ev = EVAL_BASE if i == "base-s1" else EVAL_FINAL if i == SUBMIT_ID else None
    note = NOTES.get(i, "")
    if i == SUBMIT_ID:
        note += " Đây là mô hình nộp (predictions_eval.csv)."
    rows.append(to_row(r, ev, note))
write_xlsx(rows, f"{REPO_ROOT}/templates/experiment_table_template.xlsx", f"{OUT_DIR}/experiments.xlsx",
           seed_ids=base_ids, summary_notes=SUMMARY_NOTES)
figs = sorted(f for f in os.listdir(FIG) if not f.startswith("compare_"))
assert len(figs) == len(rows) and set(figs) == {f"{r['exp_id']}.png" for r in rows}, "số ảnh phải bằng số dòng"
print(f"{len(rows)} dòng, {len(figs)} ảnh thí nghiệm, ảnh so sánh:", sorted(f for f in os.listdir(FIG) if f.startswith("compare_")))
print("Đã ghi", os.path.abspath(f"{OUT_DIR}/experiments.xlsx"))